In [11]:
# Basic imports

import os
import pandas as pd
from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent 
from langchain_google_genai import ChatGoogleGenerativeAI

#load environment variables
load_dotenv()

True

In [12]:
#load the csv file into a pandas dataframe

transactions = pd.read_csv('datasets/transactions.csv')
budget = pd.read_csv('datasets/budget.csv')
bills = pd.read_csv('datasets/bills.csv')
income = pd.read_csv('datasets/income.csv')

In [13]:
@tool
def transactions_analyzer(query: str) -> str:
    """Analyze transactions, categorize spending, and summarize expenses.

    Args:
        query : the query to check category (e.g."Food & Drinks","Entertainment") 

    Returns:
        Total amount spend on each category 
    """
    try:
        transactions['Amount'] = pd.to_numeric(transactions['Amount'], errors='coerce')

        # category mapping
        category_map = {
            "Food":"Food & Drinks",
            "Drink":"Food & Drinks",
            "Entertainment":"Entertainment",
            "Utilities":"Utilities",
            "Groceries":"Groceries",
            "Transport":"Transport"    
        }

        # look for matching category in the query
        matched_category = None
        for keyword , category_name in category_map.items():
            if keyword.lower() in query.lower():
                matched_category = category_name
                break
        if matched_category:
            total = transactions[transactions["Category"] == matched_category]['Amount'].sum()
            return f"Total for {matched_category} is: ${total:.2f}"
        return "Query not understood"
    except Exception as e:
        return f"Error accessing transactions{str(e)}"


In [ ]:
import datetime


@tool
def bill_tool(query:str) -> str:
    """"REQUIRED TOOL.
    Use this tool for any question about:
    - bills
    - due payments
    - unpaid bills
    - paid bills
    - upcoming bills
    - reminders

    Returns real bill data from CSV.
    Always call this tool.
    """
    try:

        bills['Amount'] = pd.to_numeric(bills['Amount'],errors='coerce')
        bills['Due Date'] = pd.to_datetime(bills['Due Date'],errors='coerce')
        today = datetime.datetime.now()
        q = query.lower()

        if "unpaid" in q:
            unpaid_bills = bills[bills['Status'] == 'Unpaid']['Amount'].sum()
            bill_names = bills[bills['Status'] == 'Unpaid']['Bill Name'].tolist()
            return f"Total unpaid bills:${unpaid_bills:.2f}. Unpaid bills: {','.join(bill_names)}"

        elif "paid" in q:
            paid_bills = bills[bills['Status'] == 'Paid']['Amount'].sum()
            bill_names = bills[bills['Status'] == 'Paid']['Bill Name'].tolist()
            return f"Total paid bills: ${paid_bills:.2f}. Paid bills: {', '.join(bill_names)}"
        
        elif "week" in q:
            week_end = today + datetime.timedelta(days=7)
            result = bills[(bills['Due Date'] >= today) & (bills['Due Date'] <= week_end)]
            return f"Upcoming bills in the next week: {result.to_string(index=False)}"
        
        elif "month" in q:
            result = bills[bills['Due Date'].dt.month == today.month]
            return f"Upcoming bills in the current month: {result.to_string(index=False)}"
        else:
            return "Query not understood..Specify 'paid bills', 'unpaid bills', 'bills due this week' or 'bills due this month'"
    except Exception as e:
        return f"Error accessing bills: {str(e)}"


In [15]:
@tool
def budget_tracker(query:str) -> str:
    """Check the budget for each category

    Args : the query to check category (e.g."Food & Drinks","Entertainment")

    Returns:
        Category names and total amount of the each budget category 
    """

    try:
        budget['Budget Amount'] = pd.to_numeric(budget['Budget Amount'],errors='coerce')

        category_map = {
            "Food":"Food & Drinks",
            "Drink":"Food & Drinks",
            "Entertainment":"Entertainment",
            "Utilities":"Utilities",
            "Groceries":"Groceries",
            "Transport":"Transport" 
        }

        matched_category = None
        for keyword,category_name in category_map.items():
            if keyword.lower() in query.lower():
                matched_category = category_name
                break
        if matched_category:
            budgeted_amount = budget[budget['Category'] == matched_category]['Budget Amount'].sum()
            actual_spending = transactions[transactions['Category'] == matched_category]['Amount'].sum()
            return f"Budget for {matched_category} is: ${budgeted_amount: .2f}, Actual spending is: ${actual_spending: .2f}."
        return "Query not understood"

    except Exception as e:
        return f"Error accessing budget: {str(e)}"

In [ ]:
@tool
def income_tool(query:str) -> str:
    """Check income totals, salary, sources and earnings
    Args:
        query : question about income
            examples : 
            "monthly income"
            "yearly income"
            "income sources"
            "freelance income"
    Returns:
        income summary, totals, or breakdown
    """
    try:
        
        today = datetime.datetime.now()
        q = query.lower()
        income['Amount'] = pd.to_numeric(income['Amount'], errors='coerce')
        income["Start Date"] = pd.to_datetime(income["Start Date"], errors='coerce')
        income["End Date"] = pd.to_datetime(income["End Date"], errors='coerce')
        active_income = income[(income['Start Date'] <= today) & (income['End Date'] >= today)]

        # list income sources 
        if "source" in q:
            sources = active_income['Source'].tolist()
            return f"Active income sources: {', '.join(sources)}"
        
        # monthly income
        elif "month" in q:
            total = 0
            for _,row in active_income.iterrows():
                if row['Frequency'] == "Monthly":
                    total = total + row['Amount'] 
                elif row['Frequency'] == "Weekly":
                    total = total + row['Amount'] * 4
                elif row['Frequency'] == "Quarterly":
                    total = total + row['Amount'] / 3
            return f"Estimated monthly income: ${total: .2f}"
        
        # yearly income
        elif "year" in q:
            total = 0
            for _,row in active_income.iterrows():
                if row['Frequency'] == "Monthly":
                    total = total + row['Amount'] * 12
                elif row['Frequency'] == "Weekly":
                    total = total + row['Amount'] * 52
                elif row['Frequency'] == "Quarterly":
                    total = total + row['Amount'] * 4
            return f"Estimated yearly income is: ${total: .2f}"

        # default income
        elif "total income" in q:
            total_income = active_income['Amount'].sum()
            return f"Total active income: ${total_income: .2f}"
        
        # specific source income
        else:
            for source in active_income['Source'].unique():
                if source.lower() in q:
                    amount = active_income[active_income['Source'] == source]['Amount'].sum()
                    return f"{source} income is: {amount: .2f}"
                
    except Exception as e:
        print("Error:", e)
        return f"Error accessing income data: {str(e)}"

In [17]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key = os.getenv("GEMINI-API-KEY"),
    max_tokens = 1024,
    temperature = 0
    )

In [18]:
tools =[transactions_analyzer,bill_tool,budget_tracker,income_tool]

In [19]:
# create the agent
agent = create_agent(
    model=llm,
    tools=tools
)

In [21]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "show my unpaid bills"}
    ]
})

print(response["messages"][-1].content)


[{'type': 'text', 'text': 'Total unpaid bills: $13223.87.\nUnpaid bills include: Water, Gas, Rent, Insurance, Electricity.', 'extras': {'signature': 'CrcCAb4+9vsE97b6+6kJbf9y7LT1ydH4JVTjxYmQPNwd4s1yshWMVml3e5ooG4XTi2gd1fPhChmTDqR7lYCr4YemVjD90yp0sPTCjsFNcagL8wbZwLnZmhyjqwrnwqjx5zD9t2CKUU9la6nLlYZkjGa4pfxrqhS+Ab7Pql802t8T1v75vUeAZzHCZzntzrZoS7QX8WMKZ91EWfHZwZwGyOEtBZPJUjOQuEM0MF20+FmCj0DH8q7gOpsI6pc47cCtaSMes6ufF4lESWayPIw7Smk0IvzMkwcMQZKor6Tg7G7ZC6Q+9lC5DMNwHD6Ra0ErFgvKF13OpfpzB2m5HMJvzLVvTnrlqDHy6coW8DmCLvwA0tn4z0uOWywNpVzMcm/bxI0cOvWQ3rcaBIQ4Zn9tt19SRQVwdx4rzDA='}}]
